# DAY 2：数据预处理 - 今天我们来做数据预处理。

目标是：**把产品描述重写成标准格式**。这一步非常关键。

大家还记得吗，第一天我们从亚马逊爬数据，那个 raw data 格式是乱的——标题、规格、价格全都混在一起。LLM 在这个环节能派上大用场。具体怎么操作呢？我们会给 LLM 一个 prompt，让它把每个产品描述转换成统一的结构化格式。比如：- 产品名称- 品牌- 关键特性- 类别标签这样做的好处是什么？到了后面的训练阶段，你的模型看到的是干净、一致的数据，而不是七零八落的原始文本。这听起来是脏活累活，但它直接决定了你最终模型的性能上限。所以认真对待。

<img src="../business.jpg" width="150" height="150" /> | **Business value of Data Pre-processing / Re-writing**<br><br>LLM让一些几年前还被认为是完全不可能的事情变得简单了。这种方法几乎可以应用于任何商业领域，而且它跟我们第五周使用的那些高级技术非常相似。 |

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
import requests
from pricer.siliconflow_batch import SiliconFlowBatch
from pricer.items import Item

load_dotenv(override=True)

True

下一个单元格就是让你选择数据集的地方。如果想要免费、快速版本，训练数据量 2 万条，就设置 `LITE_MODE = True`
如果想要完整、功能更强的版本，训练数据量 80 万条，就设置 `LITE_MODE = False`
## 这个实验你可以这样做
第一种，完全跳过预处理，直接从 HuggingFace 加载数据集——0 美元
第二种，只跑轻量版预处理——约 ¥17（22K 条，SiliconFlow DeepSeek-V3 Batch，**余额不足可跳过，看我演示**）
第三种，跑完整版预处理——约 ¥640（820K 条）

In [3]:
LITE_MODE = True

In [4]:
username = "iloveawscn"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='DB Electrical ADR0047 Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977 1322156 3675174RX 3920618 10459026' category='Automotive' price=111.35 full='DB Electrical  Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977   3920618 10459026\n[\'**Verify the part fits before purchasing**Specs:Unit Type: AlternatorVoltage: 24Dimension: Replacement 400-12354 Unit:, B+ Terminal: 1/4-28, Notes:, 1-Wire System, Threaded I Terminal, Combo Pin/Threaded R Terminal, 400-12354 w/Chaff Screen, AEP  wo/Chaff Screen, Amperage: 70Regulator: IRFan Type: EFBrief: Delco 21SI AlternatorUsed on Industrial, Marine and HD Truck ApplicationsReplaces OEM Numbers:: : 112160, : 107-7977, : , , , : , , , , , , , , , , , , , , : , ,  & N: 400-12081, 400-12081R, 400-12174, 400-12198, 400-12313, 400-12341, 400-12354J & N (OLD): 400-122

In [47]:
items[2].id

In [5]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [6]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [50]:
print(items[0].full)

DB Electrical  Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977   3920618 10459026
['**Verify the part fits before purchasing**Specs:Unit Type: AlternatorVoltage: 24Dimension: Replacement 400-12354 Unit:, B+ Terminal: 1/4-28, Notes:, 1-Wire System, Threaded I Terminal, Combo Pin/Threaded R Terminal, 400-12354 w/Chaff Screen, AEP  wo/Chaff Screen, Amperage: 70Regulator: IRFan Type: EFBrief: Delco 21SI AlternatorUsed on Industrial, Marine and HD Truck ApplicationsReplaces OEM Numbers:: : 112160, : 107-7977, : , , , : , , , , , , , , , , , , , , : , ,  & N: 400-12081, 400-12081R, 400-12174, 400-12198, 400-12313, 400-12341, 400-12354J & N (OLD): 400-12265, 400-: , , 1307-203-H1,  REMAN: : 7642, 8002, 8003, 8007, : ALT- LIGHT DUTY: : ALT-: 8003NCompatible With/Replacement For: 16H 1996-1999, 320B 1996-2005, 320B L 1996-2005, 320B LN 1996-2005, 320B N 1996-2005, 320B S 1996-2005, 325B L All, 325B LN All, 345B 

In [7]:
# SiliconFlow DeepSeek-V3 单条演示
# 实际调用的是 OpenAI-compatible 接口，模型换成 DeepSeek-V3（支持 Batch API，便宜）
SF_API_KEY = os.environ["SILICONCLOUD_API_KEY"]
sf_client = OpenAI(api_key=SF_API_KEY, base_url="https://api.siliconflow.cn/v1")

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = sf_client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V3",
    messages=messages,
    temperature=0.2,
    max_tokens=300,
)

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")

Title: DB Electrical 24V 70A Alternator for Industrial & Marine Use  
Category: Automotive  
Brand: DB Electrical  
Description: High-performance 24V 70A alternator compatible with various industrial and marine applications.  
Details: Features a 1-wire system, IR regulator, and EF fan type, built to OEM specifications.

Input tokens: 1429
Output tokens: 71


In [ ]:
# 如果有本地 Ollama，也可以用（需启动 ollama 服务）
try:
    from openai import OpenAI
    ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
    response = ollama_client.chat.completions.create(
        model="llama3.2",
        messages=messages,
        max_tokens=300,
    )
    print(response.choices[0].message.content)
    print(f"Input tokens: {response.usage.prompt_tokens}")
    print(f"Output tokens: {response.usage.completion_tokens}")
except Exception as e:
    print(f"Ollama 不可用: {e}")

In [8]:
MODEL = "deepseek-ai/DeepSeek-V3"

In [9]:
def make_jsonl(item):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
        "temperature": 0.2,
        "max_tokens": 300,
    }
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line, ensure_ascii=False)

In [9]:
items[0]

<DB Electrical ADR0047 Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977 1322156 3675174RX 3920618 10459026 = $111.35>

In [10]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "deepseek-ai/DeepSeek-V3", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "DB Electrical  Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977   3920618 10459026\\n[\'**Verify the part fits before purchasing**Specs:Unit Type: AlternatorVoltage: 24Dimension: Replacement 400-12354 Unit:, B+ Terminal: 1/4-28, Notes:, 1-Wire System, Threaded I Terminal, Combo Pin/Threaded R Terminal, 400-12354 w/Chaff Screen, AEP  wo/Chaff Screen, Amperage: 70Regulator: IRFan Type: EFBrief: Delco 21SI AlternatorUsed on Industrial, Marine and HD Truck Application

In [11]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [12]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [13]:
# SiliconFlow API (OpenAI-compatible)
SF_API_KEY = os.environ["SILICONCLOUD_API_KEY"]
SF_BASE = "https://api.siliconflow.cn/v1"
SF_HEADERS = {"Authorization": f"Bearer {SF_API_KEY}"}

In [14]:
# 上传 JSONL 文件到 SiliconFlow
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = requests.post(
        f"{SF_BASE}/files",
        headers=SF_HEADERS,
        files={"file": ("0_1000.jsonl", f, "application/jsonl")},
        data={"purpose": "batch"},
    )
resp_data = response.json()
file_id = resp_data.get("data", {}).get("id") or resp_data.get("id")
print(f"file_id: {file_id}")
print(f"Status: {response.status_code}")

file_id: file-3qm32txzeme27ao
Status: 200


In [15]:
file_id

'file-3qm32txzeme27ao'

In [16]:
# 创建 SiliconFlow batch
payload = {
    "input_file_id": file_id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
}
r = requests.post(f"{SF_BASE}/batches", headers={**SF_HEADERS, "Content-Type": "application/json"}, json=payload)
batch_data = r.json().get("data", r.json())
batch_id = batch_data.get("id")
print(f"batch_id: {batch_id}")
print(f"status: {batch_data.get('status')}")

batch_id: batch_xkvrrqraxw
status: in_queue


In [17]:
# 查询 batch 状态
r = requests.get(f"{SF_BASE}/batches/{batch_id}", headers=SF_HEADERS)
result = r.json().get("data", r.json())
print(f"status: {result.get('status')}")
result

status: in_progress


{'id': 'batch_xkvrrqraxw',
 'object': 'batch',
 'endpoint': '/v1/chat/completions',
 'errors': None,
 'input_file_id': 'file-3qm32txzeme27ao',
 'completion_window': '24h',
 'status': 'in_progress',
 'output_file_id': None,
 'error_file_id': None,
 'created_at': 1779594797,
 'in_progress_at': 1779594797,
 'expires_at': 1779681197,
 'finalizing_at': None,
 'completed_at': None,
 'failed_at': None,
 'expired_at': None,
 'cancelling_at': None,
 'cancelled_at': None,
 'request_counts': None,
 'metadata': None,
 'file_name': '0_1000.jsonl'}

In [19]:
# 下载结果 — SiliconFlow 的 output_file_id 是 OSS 直链 URL
output_url = result.get("output_file_id")
print(f"output_url: {output_url}")

r = requests.get(output_url)
with open("jsonl/batch_results.jsonl", "wb") as f:
    f.write(r.content)
print(f"Downloaded {len(r.content)} bytes")

output_url: None


MissingSchema: Invalid URL 'None': No scheme supplied. Perhaps you meant https://None?

In [20]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [24]:
print(items[0].full)

DB Electrical  Champion Ag & Industrial Alternator Compatible With/Replacement For Graders 710A, 716A, 5.9L Cummins 24 Volt 112160 112992 107-7977   3920618 10459026
['**Verify the part fits before purchasing**Specs:Unit Type: AlternatorVoltage: 24Dimension: Replacement 400-12354 Unit:, B+ Terminal: 1/4-28, Notes:, 1-Wire System, Threaded I Terminal, Combo Pin/Threaded R Terminal, 400-12354 w/Chaff Screen, AEP  wo/Chaff Screen, Amperage: 70Regulator: IRFan Type: EFBrief: Delco 21SI AlternatorUsed on Industrial, Marine and HD Truck ApplicationsReplaces OEM Numbers:: : 112160, : 107-7977, : , , , : , , , , , , , , , , , , , , : , ,  & N: 400-12081, 400-12081R, 400-12174, 400-12198, 400-12313, 400-12341, 400-12354J & N (OLD): 400-12265, 400-: , , 1307-203-H1,  REMAN: : 7642, 8002, 8003, 8007, : ALT- LIGHT DUTY: : ALT-: 8003NCompatible With/Replacement For: 16H 1996-1999, 320B 1996-2005, 320B L 1996-2005, 320B LN 1996-2005, 320B N 1996-2005, 320B S 1996-2005, 325B L All, 325B LN All, 345B 

In [22]:
print(items[999].summary)

Title: LED Daytime Running Lights & Halogen Fog Lights for Toyota Highlander  
Category: Automotive Lighting  
Brand: (Brand not specified)  
Description: Upgrade your 2017-2019 Toyota Highlander with bright LED daytime running lights and halogen fog lights for improved visibility.  
Details: Features switchback dual-color LEDs, waterproof ABS housing, and direct plug-and-play installation.


我把刚才说的这些逻辑封装成了一个 `SiliconFlowBatch` 类。它会自动把数据切分成每组 1,000 条，然后分别提交 Batch API，最后轮询并汇总所有结果。

**成本对比**：原来用 Groq Batch 约 ¥17（22K 条），SiliconFlow DeepSeek-V3 Batch 同样约 ¥17——价格接近，但 Groq 免费版不支持 Batch，所以改用 SiliconFlow。

> 如果余额不足，可以只看演示——Batch API 的流程是：上传 JSONL → 创建 batch → 轮询状态 → 下载结果。不需要实际运行，代码逻辑完全一样。

In [29]:
SiliconFlowBatch.create(items, LITE_MODE)

Created 22 batches


In [30]:
SiliconFlowBatch.run()

Submitting:   0%|          | 0/22 [00:00<?, ?it/s]

Submitted 22 batches, skipped 0 already submitted


In [31]:
SiliconFlowBatch.fetch()

Polling 22 pending batches...
  Batch 0_1000.jsonl: 1000 results
  Status: 1 done, 0 errors, 21 pending
  Status: 1 done, 0 errors, 21 pending
  Batch 1000_2000.jsonl: 2 quality issues:
    [1883] markdown: Title: 47" Universal Roof Cargo Basket  
Category: Automotiv
    [1898] missing_fields(Details:): Title: Custom Butterfly Acrylic Clipboard  
Category: Office
  Batch 1000_2000.jsonl: 1000 results
  Status: 2 done, 0 errors, 20 pending
  Status: 2 done, 0 errors, 20 pending
  Batch 2000_3000.jsonl: 1 quality issues:
    [2733] missing_fields(Details:): Title: Clingers Kids Dry Erase Rolls  
Category: Arts & Craf
  Batch 2000_3000.jsonl: 1000 results
  Status: 3 done, 0 errors, 19 pending
  Batch 3000_4000.jsonl: 2 quality issues:
    [3280] missing_fields(Category:,Brand:,Description:,Details:): Title: Pioneer 12" IMPP
    [3245] markdown: Title: Delphi Fuel Pump Module  
Category: Automotive  
Bran
  Batch 3000_4000.jsonl: 1000 results
  Status: 4 done, 0 errors, 18 pending
  Statu

In [32]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [33]:
print(items[10234].summary)

Title: Stainless Steel NMO Mirror Antenna Mount  
Category: Automotive  
Brand: Tram  
Description: Truck mirror mount with NMO base, 17-inch coax cable, and PL-259 connector.  
Details: Features chrome-plated brass NMO with gold contact pin and 17 ft. RG-58A/U cable.


In [34]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

最后，把最终数据集推送到 Hub。这里有两种情况：如果你选的是 lite 模式，就只推送 lite 数据集；如果是 full 模式，两个数据集都会一起推送——这样万一你之后想用 lite 版本，也还有备份可用。

In [ ]:
username = "iloveawscn"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.export_to_parquet("exported_datasets/items_lite", train=train, val=val, test=test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.export_to_parquet("exported_datasets/items_full", train=train, val=val, test=test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.export_to_parquet("exported_datasets/items_lite", train=train_lite, val=val_lite, test=test_lite)

print("Export done. Use CLI to upload:")
print(f"  hf upload-large-folder {lite} ./exported_datasets/items_lite --repo-type dataset")
if not LITE_MODE:
    print(f"  hf upload-large-folder {full} ./exported_datasets/items_full --repo-type dataset")

导出完成后，使用 CLI 上传：

```bash
# lite
hf upload-large-folder iloveawscn/items_lite ./exported_datasets/items_lite --repo-type dataset

# full（如果需要）
hf upload-large-folder iloveawscn/items_full ./exported_datasets/items_full --repo-type dataset
```

验证：
```python
from pricer.items import Item
train, val, test = Item.from_hub("iloveawscn/items_lite")
print(len(train), len(val), len(test))
```

https://huggingface.co/iloveawscn/datasets